# Inside the timber model: the graph

Notebooks 01–03 used `model.graph` in passing. Here it is the subject: open
it, see it in 3D, and ask it what a drawing answers slowly. What touches what?
How far is this beam from the ground? Where is the frame triangulated?

> **Needs the timber extra**: `uv run --extra timber jupyter lab`. Units are
> millimetres. Do `01_timber_beams_and_joints` first. Every 3D view can be
> orbited (drag), zoomed (scroll) and panned (right-drag).

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its week or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
SHARED = next(p / "Lecture" / "shared" for p in [HERE, *HERE.parents]
              if (p / "Lecture" / "shared" / "viz.py").exists())
sys.path.insert(0, str(SHARED))
from viz import explore_graph, ramp, view_timber

import tempfile
from collections import Counter, deque
from itertools import combinations

import compas
from compas.geometry import Line
from compas.tolerance import Tolerance
from compas.topology import shortest_path
from compas_timber.connections import JointTopology, LButtJoint, TButtJoint
from compas_timber.elements import Beam
from compas_timber.model import TimberModel

## Step 1 — a shed, as a model
The same shed as `03_timber_frame_to_site`. Each beam's name starts with its role.

In [ ]:
def shed(bays=2, bay=2400, depth=3600, height=2400, rise=1200, knee=800, braces=True):
    """{name: Line} for a gabled shed; the role is the part of the name before '_'."""
    length = bays * bay
    lines = {}
    for side, y in (("s", 0), ("n", depth)):
        lines[f"sill_{side}"] = Line([0, y, 0], [length, y, 0])
        lines[f"plate_{side}"] = Line([0, y, height], [length, y, height])
        for i in range(bays + 1):
            lines[f"post_{side}{i}"] = Line([i * bay, y, 0], [i * bay, y, height])
        if braces:
            lines[f"brace_{side}w"] = Line([0, y, height - knee], [knee, y, height])
            lines[f"brace_{side}e"] = Line([length, y, height - knee], [length - knee, y, height])
    for i in range(bays + 1):
        x = i * bay
        lines[f"tie_{i}"] = Line([x, 0, height], [x, depth, height])
        lines[f"rafter_s{i}"] = Line([x, 0, height], [x, depth / 2, height + rise])
        lines[f"rafter_n{i}"] = Line([x, depth, height], [x, depth / 2, height + rise])
    lines["ridge"] = Line([0, depth / 2, height + rise], [length, depth / 2, height + rise])
    return lines


SECTION = {"sill": (100, 100), "post": (100, 100), "plate": (100, 100),
           "tie": (60, 160), "rafter": (60, 160), "ridge": (60, 160), "brace": (60, 100)}
ROLE_COLOUR = {"sill": "#868e96", "post": "#1c7ed6", "plate": "#868e96", "tie": "#2f9e44",
               "rafter": "#e8590c", "ridge": "#7048e8", "brace": "#f59f00"}


def role(name):
    return name.split("_")[0]


def build(lines):
    model = TimberModel(tolerance=Tolerance(unit="MM", absolute=1e-3))
    for name, line in lines.items():
        beam = Beam.from_centerline(line, *SECTION[role(name)])
        beam.name = name
        model.add_element(beam)
    model.connect_adjacent_beams()
    return model


model = build(shed())
view_timber(model, colors={beam.name: ROLE_COLOUR[role(beam.name)] for beam in model.beams})

## Step 2 — open it up
`model.graph` is a COMPAS `Graph`, so everything from Week 06 works on it. The
model also keeps a second structure, `model.tree`: *what contains what*. The
graph says *what is connected to what*. Here the tree is flat, every beam a
child of the root, and all the structure is in the graph.

In [ ]:
graph = model.graph
print(type(graph).__name__, "is a kind of", type(graph).__mro__[1].__name__)
print(graph.number_of_nodes(), "nodes,", graph.number_of_edges(), "edges")
print(len(list(model.tree.nodes)), "tree nodes: the root and one per beam")

## Step 3 — what is stored on a node, and on an edge
A node holds a reference to its element and no real position: `x, y, z` stay
at 0. An edge holds a **candidate** (these two meet, here, like this) and, once
you decide, a **joint**. Beams know their node as `beam.graphnode`, so two small
dictionaries let us talk about beams by name.

In [ ]:
node_of = {beam.name: beam.graphnode for beam in model.beams}
name_of = {node: name for name, node in node_of.items()}

print("post_s0 is node", node_of["post_s0"])
print("node attributes:", graph.node_attributes(node_of["post_s0"]))

edge = next(iter(graph.edges()))
print("\nedge", edge, "joins", name_of[edge[0]], "and", name_of[edge[1]])
print("edge attributes:", graph.edge_attributes(edge))

TOPOLOGY = {JointTopology.TOPO_L: "L", JointTopology.TOPO_T: "T", JointTopology.TOPO_X: "X"}
candidate = graph.edge_attribute(edge, "candidates")
print("\ncandidate:", TOPOLOGY[candidate.topology], "at", candidate.location)

## Step 4 — see the graph
A node at the middle of each beam, a straight edge for each pair that meets:
orange where two ends meet (L), blue where an end meets a side (T). Orbit it
until the nodes separate.

In [ ]:
SHADE = {"L": "#e8590c", "T": "#1c7ed6"}
candidates = sorted(model.joint_candidates, key=lambda c: sorted(e.name for e in c.elements))
edge_colors = {tuple(e.name for e in c.elements): SHADE[TOPOLOGY[c.topology]] for c in candidates}

print(dict(Counter(TOPOLOGY[c.topology] for c in candidates)))
view_timber(model, graph=True, opacity=0.18, edge_colors=edge_colors, labels=["ridge", "plate_s", "tie_1"])

## Step 5 — why so many edges?
24 beams, 59 edges: much denser than the frame looks. Group the candidates by
where they are. When 4 beams meet at one point, every pair gets an edge: 6
edges, a *clique*. On a drawing that is one node of the frame. In the graph it
is six decisions.

In [ ]:
def place(candidate):
    return tuple(round(v) for v in candidate.location)


at_point = Counter(place(c) for c in candidates)
print("edges per meeting point:", dict(sorted(Counter(at_point.values()).items())))

busiest = max(sorted(at_point), key=at_point.get)
here = [tuple(e.name for e in c.elements) for c in candidates if place(c) == busiest]
beams_here = sorted({name for pair in here for name in pair})
print(f"at {busiest}: {len(beams_here)} beams, {len(here)} edges\n  {beams_here}")
view_timber(model, edges=here, opacity=0.25, labels=beams_here,
            colors={name: "#e8590c" for name in beams_here}, edge_colors={pair: "#e8590c" for pair in here})

Bending each edge through the point where the two beams meet removes the
tangle and shows where the joints are. Same graph, drawn differently.

In [ ]:
view_timber(model, graph=True, through_joints=True, opacity=0.2, edge_colors=edge_colors)

## Step 6 — ask a beam
`neighbors` and `degree`, as in Week 06. Pick a beam from the list: it turns
orange and the beams it touches turn blue.

In [ ]:
explore_graph(model)

In [ ]:
degree = {name_of[n]: graph.degree(n) for n in graph.nodes()}
for name in sorted(degree, key=degree.get, reverse=True)[:5]:
    print(f"{name:10s} degree {degree[name]}")

top = max(degree.values())
view_timber(model, colors={name: ramp(d / top) for name, d in degree.items()})

Blue touches few, orange touches many. The plates are the hubs: everything
in a long wall lands on them, so a mistake in a plate reaches the most joints.

## Step 7 — how far from the ground?
A question about **hops**, not millimetres. `shortest_path` from
`compas.topology` is the breadth-first one from Week 06: fewest edges.

In [ ]:
route = shortest_path(graph.adjacency, node_of["ridge"], node_of["sill_s"])
route = [name_of[n] for n in route]
print(" -> ".join(route), f"({len(route) - 1} hops)")
view_timber(model, edges=list(zip(route, route[1:])), opacity=0.3, labels=route,
            colors={name: "#e8590c" for name in route}, edge_colors={p: "#e8590c" for p in zip(route, route[1:])})

Run the same search from **all the sills at once** and every beam gets a
stage: how many joints it is from the foundation. Drag the slider to raise the
frame. This is a first draft of an erection sequence. The graph knows what
touches what and nothing about gravity: it puts rafters up with the plates,
because a rafter touches a post.

In [ ]:
stage = {name: 0 for name in node_of if role(name) == "sill"}
queue = deque(stage)
while queue:
    name = queue.popleft()
    for other in graph.neighbors(node_of[name]):
        if name_of[other] not in stage:
            stage[name_of[other]] = stage[name] + 1
            queue.append(name_of[other])

for s in range(max(stage.values()) + 1):
    print(f"stage {s}:", dict(Counter(role(n) for n in stage if stage[n] == s)))
view_timber(model, stages=stage, colors={name: ROLE_COLOUR[role(name)] for name in stage})

## Step 8 — where is it triangulated?
Bracing is triangles. A triangle in the graph is three beams that all touch
each other, so search for those. But a clique from Step 5 is also full of
triangles, all at one point, and those brace nothing. A **real** triangle has
its three joints at three different places: an edge attribute decides what the
shape of the graph cannot.

In [ ]:
def triangles(model):
    """(all graph triangles, the real ones), each as a sorted tuple of beam names."""
    names = {beam.graphnode: beam.name for beam in model.beams}
    where = {frozenset(e.graphnode for e in c.elements): place(c) for c in model.joint_candidates}
    found = set()
    for u, v in model.graph.edges():
        for w in set(model.graph.neighbors(u)) & set(model.graph.neighbors(v)):
            found.add(frozenset((u, v, w)))
    real = [t for t in found if len({where[frozenset(pair)] for pair in combinations(t, 2)}) == 3]
    as_names = lambda ts: sorted(tuple(sorted(names[n] for n in t)) for t in ts)
    return as_names(found), as_names(real)


found, real = triangles(model)
print(len(found), "triangles in the graph,", len(real), "real ones:")
for t in real:
    print("  ", t)
view_timber(model, colors={name: "#e8590c" for t in real for name in t})

Three roof trusses and four knee braces. Now take the braces out. Is it still
one piece? Is it still braced?

In [ ]:
def one_piece(model):
    start = next(iter(model.graph.nodes()))
    seen, queue = {start}, deque([start])
    while queue:
        for other in model.graph.neighbors(queue.popleft()):
            if other not in seen:
                seen.add(other)
                queue.append(other)
    return len(seen) == model.graph.number_of_nodes()


bare = build(shed(braces=False))
for label, m in (("with braces", model), ("without", bare)):
    print(f"{label:12s} {m.graph.number_of_nodes()} beams, one piece: {one_piece(m)}, "
          f"real triangles: {len(triangles(m)[1])}")

Connectivity does not notice the braces. Triangles do. Neither is a structural
analysis, but the second one asks the right question. **What you can get out of
a graph depends on what you store on its edges.**

## Step 9 — a joint is written on an edge
So far the edges hold only candidates. A rule turns each into a joint: the
topology picks the class, and the order of precedence from notebook 03 picks
which beam stops. After that the edge carries the joint's id, and the model
keeps the joint object.

In [ ]:
RANK = ["sill", "plate", "ridge", "post", "tie", "rafter", "brace"]      # first = runs through


def stops_first(a, b):
    return sorted((a, b), key=lambda beam: RANK.index(role(beam.name)), reverse=True)


def ends_at(a, b, tol=1.0):
    return any(b.centerline.closest_point(end).distance_to_point(end) < tol + max(b.width, b.height) / 2
               for end in (a.centerline.start, a.centerline.end))


for candidate in sorted(model.joint_candidates, key=lambda c: sorted(e.name for e in c.elements)):
    a, b = candidate.elements
    if candidate.topology == JointTopology.TOPO_T:
        TButtJoint.create(model, *((a, b) if ends_at(a, b) else (b, a)))
    elif candidate.topology == JointTopology.TOPO_L:
        LButtJoint.create(model, *stops_first(a, b))
print(len(model.process_joinery()), "joinery errors")
print(len(model.joints), "joints:", dict(Counter(type(j).__name__ for j in model.joints)))

post = model.find_element_with_name("post_s0")
print("post_s0 joints:", sorted(type(j).__name__ for j in model.get_joints_for_element(post)))
print("post_s0 machining:", [type(f).__name__.replace("Proxy", "") for f in post.features])

print("\nedges now:", graph.number_of_edges(), "| pairs of beams:", len({frozenset(e) for e in graph.edges()}))
u, v = next((u, v) for u, v in graph.edges() if graph.has_edge((v, u)))
print(f"{name_of[u]} -> {name_of[v]}: candidate {graph.edge_attribute((u, v), 'candidates') is not None}, "
      f"joint {graph.edge_attribute((u, v), 'joints') is not None}")
print(f"{name_of[v]} -> {name_of[u]}: candidate {graph.edge_attribute((v, u), 'candidates') is not None}, "
      f"joint {graph.edge_attribute((v, u), 'joints') is not None}")

More edges than before, with no new beams and no new meetings. The direction
trap from Week 06 §2: a COMPAS edge is stored in the order it was added, so
when a joint names its beams in the opposite order to the candidate, the graph
gets a second edge, and the candidate and the joint sit on different ones.
`neighbors` and `get_joints_for_element` look both ways and give the right
answer. **Counting edges does not**: count pairs of beams instead.

## Step 10 — the graph travels
Save the model and load it back in a fresh object: beams, joints and the graph
all come with it. This JSON file is the **artifact** of Week 09, the hand-off
between a tested core and whatever loads it next, Rhino included.

In [ ]:
path = pathlib.Path(tempfile.mkdtemp()) / "shed.json"
compas.json_dump(model, path)
loaded = compas.json_load(path)

print(f"{path.stat().st_size // 1024} kB")
print(type(loaded).__name__, loaded.graph.number_of_nodes(), "nodes,", len(loaded.joints), "joints")
names = {beam.graphnode: beam.name for beam in loaded.beams}
ridge = loaded.find_element_with_name("ridge")
print("ridge touches:", sorted(names[n] for n in loaded.graph.neighbors(ridge.graphnode)))

## What was all that?
`Beam.from_centerline`, `beam.length`, `model.add_element`, `TButtJoint.create`,
`beam.graphnode`. Each one is a feature of a **class**, and the graph's nodes
point at objects of those classes. Here is the family tree of two of them:

In [ ]:
for cls in (Beam, LButtJoint):
    print(" -> ".join(c.__name__ for c in cls.__mro__))

Next: [why classes, and how to write your own](../../Lecture_07/README.md).

## Your turn
1. Find every beam within 2 hops of `tie_1` and colour them in 3D.
2. Add a third bay (`shed(bays=3)`). Which numbers in Step 8 change, and which
   do not? Predict before you run it.
3. Add a brace from `post_n1` to `plate_n`. How many real triangles appear?
4. The erection stages in Step 7 put rafters up too early. Give the search a
   rule of its own: a rafter may only go up after its tie. Colour the result.